### **Data Reading**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
df_orders = spark.sql("select * from databricks_ete_project.bronze.bronze_orders")

In [0]:
df_orders.display()

In [0]:
df_orders.printSchema()

In [0]:
df_orders = df_orders.withColumn('order_date', F.to_timestamp(F.col('order_date')))
df_orders.display()

In [0]:
df_orders = df_orders.withColumn('year', F.year(F.col('order_date')))
df_orders.display()


In [0]:
df1 = df_orders.withColumn('flag', F.dense_rank().over(Window.partitionBy('year').orderBy(F.desc('total_amount'))))
df1.display()

In [0]:
df1 = df1.withColumn('rank_flag', F.rank().over(Window.partitionBy('year').orderBy(F.desc('total_amount'))))
df1.display()

In [0]:
df1 = df1.withColumn('row_flag', F.row_number().over(Window.partitionBy('year').orderBy(F.desc('total_amount'))))
df1.display()

### **Classes - OOP**

In [0]:
class Windows:

  def dense_rank(self, df):
      df_dense_rank = df.withColumn('flag', F.dense_rank().over(Window.partitionBy('year').orderBy(F.desc('total_amount'))))
      return df_dense_rank

  def rank(self, df):
      df_rank = df.withColumn('rank_flag', F.rank().over(Window.partitionBy('year').orderBy(F.desc('total_amount'))))
      return df_rank

  def row_number(self, df):
      df_row_number = df.withColumn('row_flag', F.row_number().over(Window.partitionBy('year').orderBy(F.desc('total_amount'))))        
      return df_row_number
      

In [0]:
df_new = df_orders

In [0]:
obj = Windows()
df_new = obj.dense_rank(df_new)
df_new = obj.rank(df_new)
df_new = obj.row_number(df_new)
df_new.display()

### **Data Writing**

In [0]:
df_orders.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('databricks_ete_project.silver.silver_orders')

In [0]:
%sql
select * from databricks_ete_project.silver.silver_orders